# 08 — Week 1 Assessment (Sections 15B, 15C, 15D)

Contains:
- **Exercise B:** Debugging challenge (5 cases)
- **Exercise C:** Repository review checklist (evidence populated in final close-out after verification)
- **Exercise D:** Three one-page cheat sheets (aligned with notes authoritative sources)

**From-scratch/API notes:** debugging demos may use raw tensors/autograd; any incompatible-device demo is environment-safe and caught.


In [1]:
import numpy as np
import torch

torch.manual_seed(0)
np.random.seed(0)

def get_device():
  if torch.cuda.is_available():
    return torch.device('cuda')
  if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    return torch.device('mps')
  return torch.device('cpu')

device = get_device()
print('Selected device:', device)


Selected device: cpu


## Exercise B: Debugging challenge
Diagnose and demonstrate fixes for 5 mistakes (all safe/caught where errors expected).

In [2]:
# 1. Forgetting to clear gradients
x = torch.tensor(2.0, requires_grad=True)
y = x**2
y.backward()
print('After first backward grad:', x.grad.item())
y = x**2
y.backward()
print('Bug (accumulated, forgot clear):', x.grad.item())
x.grad.zero_()
y = x**2
y.backward()
print('Fix (cleared):', x.grad.item())

# 2. Using an inappropriate learning rate
w = torch.tensor(0.0, requires_grad=True)
for i in range(5):
  loss = (w - 3)**2
  loss.backward()
  with torch.no_grad():
    w += 1.1 * w.grad  # wrong sign / too large update behavior demo
    w.grad.zero_()
  print('Step', i, 'w:', w.item(), 'loss:', loss.item())
print('Diagnosis: inappropriate LR/sign/update caused instability/divergence-like behavior. Fix: use reasonable lr and subtract gradient.')

# 3. Mixing incompatible tensor shapes
a = torch.ones(2, 3)
b = torch.ones(3, 2)
print('\nIncompatible shapes matmul/add demo:')
print('Predicted: FAIL for a+b due to shape mismatch')
try:
  _ = a + b
  print('Unexpected success')
except Exception as e:
  print('Actual exception type:', type(e).__name__)
  print('Actual exception message:', str(e))
print('Fix: align shapes, e.g. use a + b.T or reshape/broadcast appropriately:', tuple((a + b.T).shape))

# 4. Using tensors on different devices
cpu_t = torch.ones(2)
if torch.cuda.is_available():
  other = torch.ones(2, device='cuda')
  print('\nCross-device op (cpu,cuda): predicted FAIL')
  try:
    _ = cpu_t + other
  except Exception as e:
    print(type(e).__name__, str(e))
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
  other = torch.ones(2, device='mps')
  print('\nCross-device op (cpu,mps): predicted FAIL')
  try:
    _ = cpu_t + other
  except Exception as e:
    print(type(e).__name__, str(e))
else:
  print('\nCPU-only environment: unavailable-device demo (caught)')
  print('Predicted: FAIL when moving to unavailable cuda device')
  try:
    _ = cpu_t.to('cuda')
  except Exception as e:
    print(type(e).__name__, str(e))
print('Fix: move tensors to same device before operations.')

# 5. Disabling gradient tracking accidentally
x = torch.tensor(2.0, requires_grad=True)
with torch.no_grad():
  y = x**2
print('\nAccidental no_grad: y.requires_grad=', y.requires_grad)
print('Predicted: backward fails because graph not tracked')
try:
  y.backward()
except Exception as e:
  print(type(e).__name__, str(e))
print('Fix: remove unintended no_grad/detach or compute within tracked context.')


After first backward grad: 4.0
Bug (accumulated, forgot clear): 8.0
Fix (cleared): 4.0
Step 0 w: -6.600000381469727 loss: 9.0
Step 1 w: -27.720001220703125 loss: 92.16000366210938
Step 2 w: -95.30400848388672 loss: 943.7184448242188
Step 3 w: -311.5728454589844 loss: 9663.677734375
Step 4 w: -1003.6331787109375 loss: 98956.078125
Diagnosis: inappropriate LR/sign/update caused instability/divergence-like behavior. Fix: use reasonable lr and subtract gradient.

Incompatible shapes matmul/add demo:
Predicted: FAIL for a+b due to shape mismatch
Actual exception type: RuntimeError
Actual exception message: The size of tensor a (3) must match the size of tensor b (2) at non-singleton dimension 1
Fix: align shapes, e.g. use a + b.T or reshape/broadcast appropriately: (2, 3)

CPU-only environment: unavailable-device demo (caught)
Predicted: FAIL when moving to unavailable cuda device
RuntimeError Found no NVIDIA driver on your system. Please check that you have an NVIDIA GPU and installed a dr

## Exercise C: Repository review

**Evidence source (recorded during close-out):** populated after implementation/validation of notebooks 01–07 and before final execution of this notebook (08). Executed notebook outputs are persisted in-repo for auditability.

Checklist (authoritative; see `notes/001_notes.md` → `## Repository Review Checklist (Section 15C)`):

- [x] **All notebooks execute successfully**
  - Preflight: Python 3.15.0, nbformat 5.11.1, nbconvert 7.17.2, numpy 2.5.4, torch 2.15.0+cu132, matplotlib 3.11.2, Jupyter kernel available; `/tmp/opencode` exists/writable; environment CPU-only (CUDA=False, MPS=False) with CUDA→MPS→CPU fallback handling.
  - Validity: notebooks `01`–`08` are valid nbformat v4 JSON (normalized/validated).
  - Execution: notebooks executed with `python3 -m nbconvert --to notebook --execute --ExecutePreprocessor.timeout=300` from repo root in sequence `01`–`07`, Exercise C evidence materialized in `08`, then `08` executed last. (Notebook `06` had a syntax issue fixed in the accumulation-demo cell and was re-executed successfully.)
- [x] **All experiments are reproducible**
  - Seeds set where applicable (`torch.manual_seed`, `np.random.seed`), configs recorded in outputs/cells (e.g. Section 12 experiments A–F), and deterministic artifact paths used under `mlops_study_with_ai/week_01_pytorch_foundations/reports/`.
- [x] **No unexplained hardcoded assumptions exist**
  - Key constants/hyperparameters and known data-generating parameters are stated in notebook narratives/cells (true coefficients/intercepts, noise, sample sizes, learning rates, epochs, thresholds, tolerances).
- [x] **Loss curves are saved**
  - Saved artifacts in `reports/`: `section10_linear_data.png`, `section11_linear_training_loss.png`, `section12_exp_01_loss.png`–`section12_exp_06_loss.png`, `section13_classes.png`, `section13_training_loss.png`, `section14_decision_boundary.png`, `section15a_fresh_linear_regression.png`.
- [x] **Results are documented**
  - Notebooks include printed metrics/results (losses, learned vs known parameters, parity checks with explicit `allclose` tolerances, classification metrics/confusion counts, threshold comparisons) and explanatory markdown.
- [x] **README explains how to reproduce the experiments**
  - `README.md` documents actual delivered artifacts (`notes/001_notes.md`, notebooks `01`–`08`, `reports/`) and provides concrete reproduction/validation/execution steps and environment notes aligned with delivered content.

**Coverage audit note:** requested Week 01 Sections 1–15 study topics and coding exercises are mapped to canonical notes/notebooks per approved plan (including separate fresh Exercise A notebook, authoritative autograd cheat sheet, and from-scratch boundaries).


## Exercise D: Create your three cheat sheets

Aligned with `notes/001_notes.md` → `## Week 1 Cheat Sheets (Section 15D)` (authoritative sources; autograd cheat sheet identical to Section 9).

### 1. Tensor Operations and Broadcasting

```mermaid
flowchart TD
  A[Tensor ops] --> B[Creation]
  A --> C[Shape ops]
  A --> D[Math ops]
  A --> E[Reductions]
  B --> B1[tensor zeros ones randn arange linspace]
  C --> C1[reshape view transpose permute]
  C --> C2[squeeze unsqueeze cat stack]
  D --> D1[Element-wise + - * /]
  D --> D2[matmul @ dot]
  E --> E1[sum mean min max argmax argmin]
  E --> E2[Use dim and keepdim correctly]
  F[Broadcasting rules] --> F1[Align from right]
  F --> F2[Equal or size 1 dims compatible]
  F --> F3[Expand singleton dims]
  F --> F4[Otherwise incompatible error]
```

**Essentials:** track `shape/ndim/dtype/device`; reshape requires same `numel`; `cat` uses existing dim, `stack` adds new dim; reductions collapse axes (`dim` matters).

### 2. Autograd and Backpropagation (identical to Section 9 authoritative cheat sheet)

```mermaid
flowchart TD
  A[Create tensors] --> B{requires_grad?}
  B -->|Yes| C[Forward ops build graph]
  B -->|No| D[No grad history]
  C --> E[Scalar loss]
  E --> F[loss.backward]
  F --> G[.grad on leaf tensors]
  G --> H[Use gradients]
  H --> I{Need update?}
  I -->|Yes| J[torch.no_grad update]
  I -->|No| K[Inspect only]
  J --> L[Clear grads: grad.zero_ or None]
  L --> M[Next iteration]
  D --> N[torch.detach or no_grad blocks tracking]
  C --> N
  subgraph Debug
    O[Accumulation bug: missing zero_grad]
    P[Tracking bug: no_grad/detach/missing requires_grad]
    Q[Verify: analytical vs numerical vs autograd]
  end
```

**Quick rules**
- Track: set `requires_grad=True` on tensors whose gradients you need.
- Backward: call on scalar loss (or provide gradient argument for non-scalar outputs).
- Accumulation: gradients add up by default; clear before independent steps.
- Stop tracking: `torch.no_grad()` for blocks, `detach()` for tensors, `.item()` for Python scalars.
- Leaf/non-leaf: parameter/leaf grads are usually what you optimize.
- Verify: compare autograd with finite differences/analytical derivatives within tolerance.

### 3. Gradient Descent and Optimization

```mermaid
flowchart TD
  A[Init params w,b] --> B[Forward: predictions]
  B --> C[Compute loss]
  C --> D[Backward: gradients]
  D --> E[Update: param -= lr * grad]
  E --> F[Zero grads]
  F --> G{Converged or epochs done?}
  G -->|No| B
  G -->|Yes| H[Evaluate train/val]
  I[Learning rate effects] --> I1[Too small: slow convergence]
  I --> I2[Reasonable: stable convergence]
  I --> I3[Too large: divergence/instability]
  J[Diagnostics] --> J1[Loss curves]
  J --> J2[Train vs val gap]
  J --> J3[Parameter/error comparison]
```

**Essentials:** choose LR carefully; monitor train/val loss; reset grads each step; use reproducible seeds/configs; compare learned vs known parameters when ground truth exists.
